<a href="https://colab.research.google.com/github/morozovsolncev/ontology_of_synthesis/blob/main/graf_15_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ОНТОЛОГИЧЕСКАЯ МОДЕЛЬ СИНТЕЗА v4.1
Комплексный анализ с графиками и масштабированием

Генерирует данные для n = 5..50 (шаг 5), p = 0.01,0.03,0.05,0.07
Строит графики и проверяет универсальность.

Автор: Онтология синтеза
Дата: 2026
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import time
from datetime import datetime
import os
import json

# ============================================================
# КЛАСС МОДЕЛИ (базовый)
# ============================================================

class OntologySynthesisModel:
    """
    Модель роста связности в онтологии синтеза.
    Вершины — потенции, рёбра — связи, треугольники — минимальные кластеры.
    """

    def __init__(self, n_vertices, p_base, weight_mode='degree_product', random_seed=None):
        """
        Инициализация модели.

        Параметры:
        - n_vertices: количество вершин (потенций)
        - p_base: базовая вероятность образования связи
        - weight_mode: режим расчёта веса ('degree_product', 'constant', 'random')
        - random_seed: seed для воспроизводимости
        """
        self.n = n_vertices
        self.p_base = p_base
        self.weight_mode = weight_mode
        if random_seed is not None:
            np.random.seed(random_seed)
        self.seed = random_seed

        # Инициализация графа
        self.adj_matrix = np.zeros((n_vertices, n_vertices), dtype=float)
        self.weights = np.zeros((n_vertices, n_vertices), dtype=float)

        # История для записи
        self.reset_history()

    def reset_history(self):
        """Сброс истории (для нового запуска)"""
        self.history = {
            'edges': [],        # количество рёбер
            'triangles': [],    # количество треугольников
            'mean_weight': [],  # средний вес ребра
            'K_Omega': [],      # коэффициент пространственности
            'first_triangle': None,  # шаг первого треугольника
            'steps': []         # номера шагов записи
        }

    def degree(self, v):
        """Степень вершины (количество связей)"""
        return np.sum(self.adj_matrix[v] > 0)

    def product_weight(self, i, j):
        """Вес ребра как произведение степеней"""
        return self.degree(i) * self.degree(j) / (self.n * self.n)

    def constant_weight(self, i, j):
        """Постоянный вес для всех рёбер"""
        return self.p_base

    def random_weight(self, i, j):
        """Случайный вес"""
        return np.random.random() * self.p_base * 2

    def get_weight(self, i, j):
        """Выбор функции веса в зависимости от режима"""
        if self.weight_mode == 'degree_product':
            return self.product_weight(i, j)
        elif self.weight_mode == 'constant':
            return self.constant_weight(i, j)
        elif self.weight_mode == 'random':
            return self.random_weight(i, j)
        else:
            return self.product_weight(i, j)

    def count_triangles(self):
        """Подсчёт количества треугольников и их суммарной прочности"""
        triangles = 0
        strength_sum = 0.0

        for i in range(self.n):
            for j in range(i+1, self.n):
                if self.adj_matrix[i][j] > 0:
                    for k in range(j+1, self.n):
                        if self.adj_matrix[i][k] > 0 and self.adj_matrix[j][k] > 0:
                            triangles += 1
                            strength_sum += (self.weights[i][j] *
                                           self.weights[i][k] *
                                           self.weights[j][k])

        return triangles, strength_sum

    def step(self):
        """Один шаг эволюции"""
        # Выбираем случайную пару вершин
        i, j = np.random.randint(0, self.n, 2)
        if i == j:
            return

        # Текущее состояние
        has_edge = self.adj_matrix[i][j] > 0

        # Вероятность образования/усиления связи
        if has_edge:
            # Усиление существующей связи (медленнее)
            base_prob = self.p_base * 0.1
        else:
            # Образование новой связи
            base_prob = self.p_base

        # Модификация вероятности на основе степеней (чем больше связей, тем легче новые)
        degree_product = self.degree(i) * self.degree(j)
        if degree_product > 0:
            prob = base_prob * (1 + np.log1p(degree_product))
        else:
            prob = base_prob

        prob = min(prob, 1.0)  # ограничиваем

        # Акт актуализации
        if np.random.random() < prob:
            if not has_edge:
                # Создаём новую связь
                self.adj_matrix[i][j] = 1
                self.adj_matrix[j][i] = 1

            # Обновляем вес
            new_weight = self.get_weight(i, j)
            self.weights[i][j] = new_weight
            self.weights[j][i] = new_weight

    def run(self, n_steps, record_every=10, verbose=False):
        """
        Запуск модели на n_steps шагов.

        Параметры:
        - n_steps: количество шагов
        - record_every: записывать историю каждые N шагов
        - verbose: выводить прогресс

        Возвращает:
        - history: словарь с историей
        - first_triangle: шаг первого треугольника (если найден)
        """
        self.reset_history()
        first_triangle = None

        for step in range(n_steps):
            self.step()

            if step % record_every == 0:
                # Собираем статистику
                edges = np.sum(self.adj_matrix > 0) // 2
                triangles, strength = self.count_triangles()
                mean_weight = np.mean(self.weights[self.weights > 0]) if edges > 0 else 0
                K = edges * mean_weight / self.n

                # Сохраняем
                self.history['edges'].append(edges)
                self.history['triangles'].append(triangles)
                self.history['mean_weight'].append(mean_weight)
                self.history['K_Omega'].append(K)
                self.history['steps'].append(step)

                # Отмечаем первый треугольник
                if triangles > 0 and first_triangle is None:
                    first_triangle = step
                    self.history['first_triangle'] = step

                if verbose and step % (n_steps // 5) == 0:
                    print(f"    шаг {step:4d}: рёбер={edges:3d}, треуг={triangles:3d}, "
                          f"K={K:.3f}, вес={mean_weight:.3f}")

        return self.history, first_triangle


# ============================================================
# ФУНКЦИИ АНАЛИЗА И ВИЗУАЛИЗАЦИИ
# ============================================================

def analyze_run(history, n_value):
    """
    Анализ одного прогона: вычисляет ключевые метрики
    history - словарь с историей
    n_value - размер системы (нужен для расчёта плотности)
    """
    edges = np.array(history['edges'])
    triangles = np.array(history['triangles'])
    steps = np.array(history['steps'])

    metrics = {}

    # Время первого треугольника
    metrics['first_triangle_step'] = history.get('first_triangle', None)

    # Финальные значения
    metrics['final_edges'] = edges[-1] if len(edges) > 0 else 0
    metrics['final_triangles'] = triangles[-1] if len(triangles) > 0 else 0

    # Плотность при первом треугольнике
    if metrics['first_triangle_step'] is not None:
        idx = np.where(steps == metrics['first_triangle_step'])[0]
        if len(idx) > 0:
            metrics['density_at_first'] = edges[idx[0]] / n_value
        else:
            metrics['density_at_first'] = None
    else:
        metrics['density_at_first'] = None

    # Скорость роста треугольников после порога
    if len(triangles) > 10:
        # Аппроксимируем последнюю треть логарифмом треугольников
        t = triangles[-len(triangles)//3:]
        if np.all(t > 0):
            log_t = np.log(t)
            x = np.arange(len(log_t))
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, log_t)
            metrics['growth_rate'] = slope
            metrics['growth_r2'] = r_value**2
        else:
            metrics['growth_rate'] = 0
            metrics['growth_r2'] = 0
    else:
        metrics['growth_rate'] = 0
        metrics['growth_r2'] = 0

    return metrics


def plot_regime_summary(all_histories, n, p, output_dir="plots"):
    """
    Строит сводные графики для одного режима (n,p) по всем повторениям
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Собираем все кривые
    steps = all_histories[0]['steps']
    n_steps = len(steps)

    edges_all = np.zeros((len(all_histories), n_steps))
    triangles_all = np.zeros((len(all_histories), n_steps))
    K_all = np.zeros((len(all_histories), n_steps))

    for i, h in enumerate(all_histories):
        edges_all[i] = h['edges']
        triangles_all[i] = h['triangles']
        K_all[i] = h['K_Omega']

    # Средние и стандартные отклонения
    edges_mean = np.mean(edges_all, axis=0)
    edges_std = np.std(edges_all, axis=0)

    triangles_mean = np.mean(triangles_all, axis=0)
    triangles_std = np.std(triangles_all, axis=0)

    K_mean = np.mean(K_all, axis=0)
    K_std = np.std(K_all, axis=0)

    # Создаём график
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'n={n}, p={p} (среднее по {len(all_histories)} повторениям)', fontsize=16)

    # 1. Рёбра
    ax = axes[0,0]
    ax.fill_between(steps, edges_mean - edges_std, edges_mean + edges_std, alpha=0.3, color='blue')
    ax.plot(steps, edges_mean, 'b-', linewidth=2, label='среднее')
    ax.set_xlabel('шаг')
    ax.set_ylabel('количество рёбер')
    ax.set_title('Рёбра')
    ax.grid(True, alpha=0.3)

    # 2. Треугольники
    ax = axes[0,1]
    ax.fill_between(steps, triangles_mean - triangles_std, triangles_mean + triangles_std, alpha=0.3, color='red')
    ax.plot(steps, triangles_mean, 'r-', linewidth=2, label='среднее')
    ax.set_xlabel('шаг')
    ax.set_ylabel('количество треугольников')
    ax.set_title('Треугольники')
    ax.grid(True, alpha=0.3)

    # 3. K_Omega
    ax = axes[1,0]
    ax.fill_between(steps, K_mean - K_std, K_mean + K_std, alpha=0.3, color='green')
    ax.plot(steps, K_mean, 'g-', linewidth=2, label='среднее')
    ax.set_xlabel('шаг')
    ax.set_ylabel('K_Ω')
    ax.set_title('Коэффициент пространственности K_Ω')
    ax.grid(True, alpha=0.3)

    # 4. Плотность и треугольники
    ax = axes[1,1]
    density = edges_mean / n
    ax.scatter(density, triangles_mean, c='purple', s=30, alpha=0.7)
    ax.set_xlabel('плотность рёбер (E/n)')
    ax.set_ylabel('среднее число треугольников')
    ax.set_title('Фазовая диаграмма')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/regime_n{n}_p{int(p*100):02d}.png", dpi=150)
    plt.close()

    return {
        'steps': steps,
        'edges_mean': edges_mean,
        'edges_std': edges_std,
        'triangles_mean': triangles_mean,
        'triangles_std': triangles_std,
        'K_mean': K_mean,
        'K_std': K_std
    }


def plot_scaling_test(results, output_dir="plots"):
    """
    Проверка масштабирования: нормировка времени на n и p
    results: словарь {cell_key: {'n':n, 'p':p, 'steps':steps, 'triangles_mean':list}}
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Проверка универсальности (масштабирование)', fontsize=16)

    # Создаём цветовую карту
    all_n = [data['n'] for data in results.values()]
    all_p = [data['p'] for data in results.values()]
    # Уникальные комбинации для цвета
    unique_keys = list(results.keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_keys)))

    # 1. Без нормировки (все кривые вместе)
    ax = axes[0,0]
    for (key, data), color in zip(results.items(), colors):
        n = data['n']
        p = data['p']
        label = f"n={n}, p={p}"
        ax.plot(data['steps'], data['triangles_mean'], color=color, label=label, alpha=0.7)
    ax.set_xlabel('шаг')
    ax.set_ylabel('треугольники (среднее)')
    ax.set_title('Без нормировки')
    ax.grid(True, alpha=0.3)
    # Уменьшаем размер легенды, если много линий
    if len(results) > 8:
        ax.legend(loc='upper left', fontsize=6)
    else:
        ax.legend(loc='upper left', fontsize=8)

    # 2. Нормировка времени на n (предполагаем, что время пропорционально n)
    ax = axes[0,1]
    for (key, data), color in zip(results.items(), colors):
        n = data['n']
        p = data['p']
        steps_norm = np.array(data['steps']) / n
        label = f"n={n}, p={p}"
        ax.plot(steps_norm, data['triangles_mean'], color=color, label=label, alpha=0.7)
    ax.set_xlabel('шаг / n')
    ax.set_ylabel('треугольники (среднее)')
    ax.set_title('Нормировка времени на n')
    ax.grid(True, alpha=0.3)
    if len(results) > 8:
        ax.legend(loc='upper left', fontsize=6)
    else:
        ax.legend(loc='upper left', fontsize=8)

    # 3. Нормировка времени на p (предполагаем, что время обратно пропорционально p)
    ax = axes[1,0]
    for (key, data), color in zip(results.items(), colors):
        n = data['n']
        p = data['p']
        steps_norm = np.array(data['steps']) * p
        label = f"n={n}, p={p}"
        ax.plot(steps_norm, data['triangles_mean'], color=color, label=label, alpha=0.7)
    ax.set_xlabel('шаг × p')
    ax.set_ylabel('треугольники (среднее)')
    ax.set_title('Нормировка времени на p')
    ax.grid(True, alpha=0.3)
    if len(results) > 8:
        ax.legend(loc='upper left', fontsize=6)
    else:
        ax.legend(loc='upper left', fontsize=8)

    # 4. Нормировка на n×p (оба параметра)
    ax = axes[1,1]
    for (key, data), color in zip(results.items(), colors):
        n = data['n']
        p = data['p']
        steps_norm = np.array(data['steps']) * p / n
        label = f"n={n}, p={p}"
        ax.plot(steps_norm, data['triangles_mean'], color=color, label=label, alpha=0.7)
    ax.set_xlabel('шаг × p / n')
    ax.set_ylabel('треугольники (среднее)')
    ax.set_title('Нормировка на p/n')
    ax.grid(True, alpha=0.3)
    if len(results) > 8:
        ax.legend(loc='upper left', fontsize=6)
    else:
        ax.legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/scaling_test.png", dpi=150)
    plt.close()


def plot_phase_diagram(metrics_by_cell, output_dir="plots"):
    """
    Строит фазовую диаграмму (n, p) -> треугольники
    """
    n_values = sorted(set(m['n'] for m in metrics_by_cell.values()))
    p_values = sorted(set(m['p'] for m in metrics_by_cell.values()))

    # Создаём матрицу для тепловой карты
    triangles_matrix = np.zeros((len(n_values), len(p_values)))
    first_step_matrix = np.zeros((len(n_values), len(p_values)))

    for i, n in enumerate(n_values):
        for j, p in enumerate(p_values):
            # Ищем ключ вида "n{p}_p{p}"
            found = False
            for key, m in metrics_by_cell.items():
                if m['n'] == n and abs(m['p'] - p) < 0.001:
                    triangles_matrix[i,j] = m['triangles_mean']
                    first_step_matrix[i,j] = m.get('first_triangle_mean', 0)
                    found = True
                    break
            if not found:
                triangles_matrix[i,j] = 0
                first_step_matrix[i,j] = 0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Тепловая карта треугольников
    ax = axes[0]
    im = ax.imshow(triangles_matrix, origin='lower', aspect='auto', cmap='viridis',
                   extent=[min(p_values), max(p_values), min(n_values), max(n_values)])
    ax.set_xlabel('p')
    ax.set_ylabel('n')
    ax.set_title('Среднее число треугольников')
    plt.colorbar(im, ax=ax)

    # Тепловая карта времени первого треугольника
    ax = axes[1]
    im = ax.imshow(first_step_matrix, origin='lower', aspect='auto', cmap='plasma',
                   extent=[min(p_values), max(p_values), min(n_values), max(n_values)])
    ax.set_xlabel('p')
    ax.set_ylabel('n')
    ax.set_title('Время первого треугольника')
    plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/phase_diagram.png", dpi=150)
    plt.close()


# ============================================================
# ОСНОВНОЙ ЭКСПЕРИМЕНТ
# ============================================================

def run_comprehensive_experiment():
    """
    Запускает полный эксперимент:
    - n от 5 до 50 с шагом 5
    - p = 0.01, 0.03, 0.05, 0.07
    - 10 повторений для каждой ячейки
    - 500 шагов
    """
    print("\n" + "="*70)
    print("КОМПЛЕКСНОЕ ИССЛЕДОВАНИЕ ОНТОЛОГИЧЕСКОЙ МОДЕЛИ v4.0")
    print("="*70)

    # Параметры
    n_values = list(range(5, 51, 5))
    p_values = [0.01, 0.03, 0.05, 0.07]
    repeats = 10
    steps = 3000
    record_every = 10

    print(f"n: {n_values}")
    print(f"p: {p_values}")
    print(f"повторений: {repeats}")
    print(f"всего симуляций: {len(n_values) * len(p_values) * repeats}")
    print(f"шагов: {steps}\n")

    # Структура для результатов
    all_data = {}          # сырые истории
    metrics_by_cell = {}   # метрики по ячейкам

    total = len(n_values) * len(p_values) * repeats
    current = 0
    start_time = time.time()

    # Создаём директории
    data_dir = "data"
    plots_dir = "plots"
    for d in [data_dir, plots_dir]:
        if not os.path.exists(d):
            os.makedirs(d)

    # Основной цикл
    for n in n_values:
        for p in p_values:
            cell_key = f"n{n}_p{p}"
            print(f"\n--- Ячейка: {cell_key} ---")

            histories = []
            first_triangles = []

            for rep in range(repeats):
                current += 1
                seed = hash(f"comprehensive_{n}_{p}_{rep}") % (2**32)

                model = OntologySynthesisModel(n, p, random_seed=seed)
                history, first = model.run(steps, record_every, verbose=False)

                # Конвертируем numpy-типы
                clean_history = {
                    'edges': [int(x) for x in history['edges']],
                    'triangles': [int(x) for x in history['triangles']],
                    'mean_weight': [float(x) for x in history['mean_weight']],
                    'K_Omega': [float(x) for x in history['K_Omega']],
                    'steps': [int(x) for x in history['steps']],
                    'first_triangle': int(first) if first is not None else None
                }
                histories.append(clean_history)

                if first is not None:
                    first_triangles.append(first)

                if (rep+1) % 5 == 0:
                    print(f"  повтор {rep+1}/{repeats} [{current}/{total}]")

            # Сохраняем истории
            all_data[cell_key] = histories

            # Вычисляем метрики для ячейки
            metrics_list = []
            for h in histories:
                m = analyze_run(h, n)  # передаём n явно
                # Добавляем n и p для контекста
                m['n'] = n
                m['p'] = p
                metrics_list.append(m)

            # Усредняем по повторениям
            cell_metrics = {
                'n': n,
                'p': p,
                'triangles_mean': float(np.mean([m['final_triangles'] for m in metrics_list])),
                'triangles_std': float(np.std([m['final_triangles'] for m in metrics_list])),
                'edges_mean': float(np.mean([m['final_edges'] for m in metrics_list])),
                'edges_std': float(np.std([m['final_edges'] for m in metrics_list])),
                'first_triangle_mean': float(np.mean([m['first_triangle_step'] for m in metrics_list if m['first_triangle_step'] is not None])) if any(m['first_triangle_step'] is not None for m in metrics_list) else None,
                'first_triangle_std': float(np.std([m['first_triangle_step'] for m in metrics_list if m['first_triangle_step'] is not None])) if any(m['first_triangle_step'] is not None for m in metrics_list) else None,
                'growth_rate_mean': float(np.mean([m['growth_rate'] for m in metrics_list])),
                'growth_rate_std': float(np.std([m['growth_rate'] for m in metrics_list])),
                'density_at_first_mean': float(np.mean([m['density_at_first'] for m in metrics_list if m['density_at_first'] is not None])) if any(m['density_at_first'] is not None for m in metrics_list) else None,
                'density_at_first_std': float(np.std([m['density_at_first'] for m in metrics_list if m['density_at_first'] is not None])) if any(m['density_at_first'] is not None for m in metrics_list) else None,
            }
            metrics_by_cell[cell_key] = cell_metrics

            # Строим сводный график для этого режима
            plot_regime_summary(histories, n, p, plots_dir)

    elapsed = time.time() - start_time
    print(f"\nЭксперимент завершён за {elapsed:.1f} сек")

    # Сохраняем сырые данные
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    with open(f"{data_dir}/all_data_{timestamp}.json", 'w') as f:
        json.dump(all_data, f, indent=2)

    # Сохраняем метрики
    with open(f"{data_dir}/metrics_{timestamp}.json", 'w') as f:
        json.dump(metrics_by_cell, f, indent=2)

    # Строим проверку масштабирования
    # Для этого нам нужны усреднённые кривые по каждой ячейке
    scaling_data = {}
    for cell_key, histories in all_data.items():
        n = int(cell_key.split('_')[0][1:])
        p = float(cell_key.split('_')[1][1:])

        # Усредняем кривые
        steps = histories[0]['steps']
        triangles_all = np.array([h['triangles'] for h in histories])
        triangles_mean = np.mean(triangles_all, axis=0)

        scaling_data[cell_key] = {
            'n': n,
            'p': p,
            'steps': steps,
            'triangles_mean': triangles_mean.tolist()
        }

    plot_scaling_test(scaling_data, plots_dir)

    # Строим фазовую диаграмму
    plot_phase_diagram(metrics_by_cell, plots_dir)

    # Выводим сводную таблицу
    print("\n" + "="*70)
    print("СВОДНАЯ ТАБЛИЦА МЕТРИК")
    print("="*70)
    print(f"{'n':>3} {'p':>5} {'треуг':>8} {'±σ':>6} {'первый':>8} {'плотн1':>8} {'рост':>8}")
    print("-"*70)

    for n in n_values:
        for p in p_values:
            key = f"n{n}_p{p}"
            if key in metrics_by_cell:
                m = metrics_by_cell[key]
                triangles_str = f"{m['triangles_mean']:8.1f} ±{m['triangles_std']:5.1f}"
                first_str = f"{m['first_triangle_mean']:8.1f}" if m['first_triangle_mean'] is not None else "     —"
                density_str = f"{m['density_at_first_mean']:8.3f}" if m['density_at_first_mean'] is not None else "     —"
                growth_str = f"{m['growth_rate_mean']:8.3f}"
                print(f"{n:3d} {p:5.2f} {triangles_str} {first_str} {density_str} {growth_str}")

    print(f"\nГрафики сохранены в {plots_dir}/")
    print(f"Данные сохранены в {data_dir}/")

    return all_data, metrics_by_cell


# ============================================================
# ЗАПУСК
# ============================================================

if __name__ == "__main__":
    all_data, metrics = run_comprehensive_experiment()

    print("\n" + "="*70)
    print("ГОТОВО! Все графики построены, данные сохранены.")
    print("="*70)


КОМПЛЕКСНОЕ ИССЛЕДОВАНИЕ ОНТОЛОГИЧЕСКОЙ МОДЕЛИ v4.0
n: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
p: [0.01, 0.03, 0.05, 0.07]
повторений: 10
всего симуляций: 400
шагов: 3000


--- Ячейка: n5_p0.01 ---
  повтор 5/10 [5/400]
  повтор 10/10 [10/400]

--- Ячейка: n5_p0.03 ---
  повтор 5/10 [15/400]
  повтор 10/10 [20/400]

--- Ячейка: n5_p0.05 ---
  повтор 5/10 [25/400]
  повтор 10/10 [30/400]

--- Ячейка: n5_p0.07 ---
  повтор 5/10 [35/400]
  повтор 10/10 [40/400]

--- Ячейка: n10_p0.01 ---
  повтор 5/10 [45/400]
  повтор 10/10 [50/400]

--- Ячейка: n10_p0.03 ---
  повтор 5/10 [55/400]
  повтор 10/10 [60/400]

--- Ячейка: n10_p0.05 ---
  повтор 5/10 [65/400]
  повтор 10/10 [70/400]

--- Ячейка: n10_p0.07 ---
  повтор 5/10 [75/400]
  повтор 10/10 [80/400]

--- Ячейка: n15_p0.01 ---
  повтор 5/10 [85/400]
  повтор 10/10 [90/400]

--- Ячейка: n15_p0.03 ---
  повтор 5/10 [95/400]
  повтор 10/10 [100/400]

--- Ячейка: n15_p0.05 ---
  повтор 5/10 [105/400]
  повтор 10/10 [110/400]

--- Ячейка: n1